# GEMASTIK XVII final: UKT group classification with CatBoost / LightGBM / XGBoost OOF + hill-climbing ensemble

In [5]:
import pandas as pd
import numpy as np
import time
import pickle as pkl
import re
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score

from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import optuna
# from pycaret.classification import *
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.features.generators import AutoMLPipelineFeatureGenerator
# from tabpfn import TabPFNClassifier
# from pytorch_tabnet.tab_model import TabNetClassifier

In [3]:

# Train models
def multiclass_oof_preds(
    classifier,
    model_key,
    X_train,
    y_train,
    X_test,
    metric_score,
    metric_score_args={},
    model_args={},
    fitting_args={},
    n_folds=5,
    dir_name="output",
    verbose=False,
):
    # if X_test != None:
    test_preds = np.zeros((len(X_test), y_train.nunique()))
    # else:
    #     test_preds = np.array([])
    oof_full = np.zeros((len(y_train), y_train.nunique()))
    is_target_str = y_train.dtype == "object"

    # Start timer
    start = time.time()

    # k-fold cross validation
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=0)

    score = 0
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
        # Get training and validation sets
        x_tr, x_valid = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_valid = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Start timer
        start = time.time()

        # # Tune hyperparameters
        # clf = RandomizedSearchCV(estimator=classifier, \
        #                          param_distributions=grid[model_key], \
        #                          n_iter=20, scoring='average_precision', n_jobs=-1, cv=5)

        # Train using PredefinedSplit
        clf = classifier(
            **model_args
        )
        clf.fit(x_tr, y_tr, 
                # sample_weight=sample_weights[train_idx],
                **fitting_args)

        # Out-of-fold predictions
        oof_preds = clf.predict_proba(x_valid)
        if is_target_str:
            score += (
                metric_score(
                    y_valid.tolist(),
                    [clf.classes_[i] for i in np.argmax(oof_preds, axis=1)],
                    **metric_score_args,
                )
                / n_folds
            )
        else:
            score += (
                metric_score(y_valid, np.argmax(oof_preds, axis=1), **metric_score_args)
                / n_folds
            )
        oof_full[val_idx] = oof_preds

        # if X_test != None:
            # Test set predictions
        test_preds += clf.predict_proba(X_test) / n_folds

    # Stop timer
    stop = time.time()

    if verbose:
        # Print score and time
        print("Model:", model_key)
        print("Average Metric:", np.round(100 * score, 2))
        print("Training time (mins):", np.round((stop - start) / 60, 2))
        print("")

    # Save oof and test set preds
    with open(f"{dir_name}/{model_key}_oof_preds.pkl", "wb") as f:
        pkl.dump(oof_full, f)
    # ss.to_csv(f"{dir_name}/{model_key}_test_preds.csv", index=False)
    # if X_test != None:
    with open(f"{dir_name}/{model_key}_test_preds.pkl", "wb") as f:
        pkl.dump(test_preds, f)

    return score, oof_full, test_preds

In [3]:
train_df = pd.read_csv("dataset/train.csv")
# train_df = pd.read_csv('our_dataset/train_clean_with_outliers.csv')
train_provinsi = pd.read_csv('our_dataset/train_provinsi.csv')
train_df.head()

,id,Pendapatan Keluarga,Jumlah Tanggungan dalam Keluarga,Pekerjaan Orang Tua/Wali,Pendidikan Orang Tua/Wali,Kepemilikan Aset,Lokasi Tempat Tinggal,Pengeluaran Bulanan Keluarga,Status Kepesertaan Program Bantuan Pemerintah,Riwayat Beasiswa atau Bantuan Pendidikan,...,Bahan Atap Rumah,Bahan Tembok Rumah,Bahan Lantai Rumah,Rerata Pengeluaran Listrik & Air per Bulan,Total Hutang,Cicilan Hutang per Bulan,Program Studi,Kota,Jalur Masuk,Golongan UKT
0,127748,Rp 1.000.000 s.d. Rp 2.000.000,2.0,Tidak Bekerja,SLTA Sederajat,Milik Sendiri,Kab. Sragen,500000.0,KIS/JKS/BPJS,KIPK,...,genting,semen,keramik,450000.0,0.0,0.0,Ilmu Politik,Kab. Sragen,SNBT,0.0
1,90205,lebih dari Rp 5.000.000,2.0,Buruh,SLTA Sederajat,Milik Sendiri,Kabupaten Tangerang,3500000.0,KIS/JKS/BPJS,Tidak Ada,...,Genteng,Batako Plester,Keramik,320000.0,45000000.0,4500000.0,Geografi,Kabupaten Tangerang,SNBP,3.0
2,82795,Rp 3.000.000 s.d. Rp 4.000.000,2.0,Wiraswasta,SD,Milik Sendiri,Kabupaten Grobogan,600000.0,KIS/JKS/BPJS,Tidak,...,Genting,Batu Bata,Keramik,NaN,0.0,0.0,Ilmu Politik,Kabupaten Grobogan,SNBT,4.0
3,108947,lebih dari Rp 5.000.000,2.0,Wiraswasta,SLTP,Milik Sendiri,Kab. Bogor,2250000.0,KIS/JKS/BPJS,Tidak Dapat,...,Kayu dan asbes,Bata merah dan hebel,Keramik,500000.0,0.0,0.0,Pendidikan Ekonomi,Kab. Bogor,SNBT,5.0
4,89985,Rp 3.000.000 s.d. Rp 4.000.000,NaN,Wiraswasta,MTs,Milik Sendiri,Kabupaten Pemalang,2310000.0,KIS/JKS/BPJS,Tidak Dapat,...,"Genting tanah liat, asbes, dan seng.",Batu bata,"Keramik, tegel, plester.",105000.0,5000000.0,500000.0,Pendidikan Teknik Elektro,Kabupaten Pemalang,SNBP,3.0


In [206]:
train_raw_df = pd.read_csv("dataset/train.csv")
# train_raw_df = train_raw_df[train_raw_df.id.isin(train_df.id)]
train_raw_df.shape

(30380, 28)

In [5]:
test_df = pd.read_csv("dataset/test.csv")
test_provinsi = pd.read_csv("our_dataset/train_provinsi.csv")

In [209]:
train_df["Provinsi"] = train_provinsi["Provinsi"]
test_df["Provinsi"] = test_provinsi["Provinsi"]

In [210]:
X, y = train_df.drop(["Golongan UKT"], axis=1), train_df["Golongan UKT"]

In [212]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30380 entries, 0 to 30379
Data columns (total 28 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   id                                                      30380 non-null  int64  
 1   Pendapatan Keluarga                                     30168 non-null  object 
 2   Jumlah Tanggungan dalam Keluarga                        30160 non-null  float64
 3   Pekerjaan Orang Tua/Wali                                30186 non-null  object 
 4   Pendidikan Orang Tua/Wali                               30180 non-null  object 
 5   Kepemilikan Aset                                        30173 non-null  object 
 6   Lokasi Tempat Tinggal                                   30237 non-null  object 
 7   Pengeluaran Bulanan Keluarga                            30159 non-null  float64
 8   Status Kepesertaan Program Bantuan P

In [213]:
text_cols = ["Bahan Atap Rumah", "Bahan Tembok Rumah", "Bahan Lantai Rumah"]

In [214]:
cat_cols = list(set(X.columns[X.dtypes == object].tolist() + X.iloc[:, 22:].columns.tolist()))
num_cols = list(set(X.columns.tolist()) - set(cat_cols))
print(cat_cols)
print(num_cols)

['Status Kepesertaan Program Bantuan Pemerintah', 'Lokasi Tempat Tinggal', 'Status Kepemilikan Tempat Tinggal', 'Bahan Tembok Rumah', 'Pekerjaan Orang Tua/Wali', 'Kepemilikan Aset', 'Program Studi', 'Bahan Atap Rumah', 'Cicilan Hutang per Bulan', 'Bahan Lantai Rumah', 'Riwayat Beasiswa atau Bantuan Pendidikan', 'Pendapatan Keluarga', 'Pendidikan Orang Tua/Wali', 'Ayah Hidup?', 'Total Hutang', 'Ibu Hidup?', 'Jalur Masuk', 'Provinsi', 'Kota']
['Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki', 'Pengeluaran Bulanan Keluarga', 'Luas Tanah Rumah (m2)', 'NJOP per Meter', 'id', 'Luas Bangunan Rumah (m2)', 'Rerata Pengeluaran Listrik & Air per Bulan', 'Jumlah Tanggungan dalam Keluarga', 'Daya Listrik (Watt)']


In [215]:
X_filled = X.copy()
X_filled[cat_cols].fillna("Yang Lain", inplace=True)
X_filled.fillna(-1, inplace=True)

In [216]:
autogluon_dfx = TabularDataset(X_filled)
autogluon_dfx.head()

,id,Pendapatan Keluarga,Jumlah Tanggungan dalam Keluarga,Pekerjaan Orang Tua/Wali,Pendidikan Orang Tua/Wali,Kepemilikan Aset,Lokasi Tempat Tinggal,Pengeluaran Bulanan Keluarga,Status Kepesertaan Program Bantuan Pemerintah,Riwayat Beasiswa atau Bantuan Pendidikan,...,Bahan Atap Rumah,Bahan Tembok Rumah,Bahan Lantai Rumah,Rerata Pengeluaran Listrik & Air per Bulan,Total Hutang,Cicilan Hutang per Bulan,Program Studi,Kota,Jalur Masuk,Provinsi
0,127748,Rp 1.000.000 s.d. Rp 2.000.000,2.0,Tidak Bekerja,SLTA Sederajat,Milik Sendiri,Kab. Sragen,500000.0,KIS/JKS/BPJS,KIPK,...,genting,semen,keramik,450000.0,0.0,0.0,Ilmu Politik,Kab. Sragen,SNBT,Jawa Tengah (JATENG)
1,90205,lebih dari Rp 5.000.000,2.0,Buruh,SLTA Sederajat,Milik Sendiri,Kabupaten Tangerang,3500000.0,KIS/JKS/BPJS,Tidak Ada,...,Genteng,Batako Plester,Keramik,320000.0,45000000.0,4500000.0,Geografi,Kabupaten Tangerang,SNBP,Banten
2,82795,Rp 3.000.000 s.d. Rp 4.000.000,2.0,Wiraswasta,SD,Milik Sendiri,Kabupaten Grobogan,600000.0,KIS/JKS/BPJS,Tidak,...,Genting,Batu Bata,Keramik,-1.0,0.0,0.0,Ilmu Politik,Kabupaten Grobogan,SNBT,Jawa Tengah (JATENG)
3,108947,lebih dari Rp 5.000.000,2.0,Wiraswasta,SLTP,Milik Sendiri,Kab. Bogor,2250000.0,KIS/JKS/BPJS,Tidak Dapat,...,Kayu dan asbes,Bata merah dan hebel,Keramik,500000.0,0.0,0.0,Pendidikan Ekonomi,Kab. Bogor,SNBT,Jawa Barat (JABAR)
4,89985,Rp 3.000.000 s.d. Rp 4.000.000,-1.0,Wiraswasta,MTs,Milik Sendiri,Kabupaten Pemalang,2310000.0,KIS/JKS/BPJS,Tidak Dapat,...,"Genting tanah liat, asbes, dan seng.",Batu bata,"Keramik, tegel, plester.",105000.0,5000000.0,500000.0,Pendidikan Teknik Elektro,Kabupaten Pemalang,SNBP,Jawa Tengah (JATENG)


In [217]:
auto_ml_pipeline_feature_generator = AutoMLPipelineFeatureGenerator()
X_gluon = auto_ml_pipeline_feature_generator.fit_transform(X=autogluon_dfx)
X_gluon

id  Jumlah Tanggungan dalam Keluarga  Pengeluaran Bulanan Keluarga  \
0      127748                               2.0                      500000.0   
1       90205                               2.0                     3500000.0   
2       82795                               2.0                      600000.0   
3      108947                               2.0                     2250000.0   
4       89985                              -1.0                     2310000.0   
...       ...                               ...                           ...   
30375  147514                               0.0                      500000.0   
30376   62832                               0.0                      600000.0   
30377   88943                               3.0                      450000.0   
30378  149599                               1.0                      450000.0   
30379   82699                               2.0                     1100000.0   

       Status Kepesertaan Program Bantuan Pemerintah  \
0                                                  0   
1                                                  0   
2                                                  0   
3                                                  0   
4                                                  0   
...                                              ...   
30375                                              0   
30376                                              1   
30377                                              0   
30378                                              0   
30379                                              0   

       Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki  \
0                                                    2.0        
1                                              6500000.0        
2                                             20000000.0        
3                                              8000000.0        
4                                              1000000.0        
...                                                  ...        
30375                                         18000000.0        
30376                                          1000000.0        
30377                                          3000000.0        
30378                                          6000000.0        
30379                                          4500000.0        

       Daya Listrik (Watt)  Luas Tanah Rumah (m2)  Luas Bangunan Rumah (m2)  \
0                    900.0                  144.0                     126.0   
1                    900.0                   60.0                      22.0   
2                    450.0                  586.0                      65.0   
3                    900.0                   93.0                      70.0   
4                    900.0                    0.0                      57.0   
...                    ...                    ...                       ...   
30375                900.0                 2700.0                      90.0   
30376                450.0                  142.0                      42.0   
30377                450.0                  450.0                      40.0   
30378                450.0                  310.0                      40.0   
30379                450.0                  441.0                      54.0   

       NJOP per Meter  Rerata Pengeluaran Listrik & Air per Bulan  ...  \
0               103.0                                    450000.0  ...   
1               128.0                                    320000.0  ...   
2            128000.0                                        -1.0  ...   
3           2176000.0                                    500000.0  ...   
4                 0.0                                    105000.0  ...   
...               ...                                         ...  ...   
30375        160000.0                                     60000.0  ...   
30376         36000.0                      

In [218]:
train_df = auto_ml_pipeline_feature_generator.transform(X=autogluon_dfx)

In [219]:
list(set(X_filled.columns) - set(X_gluon.columns))

[]

In [220]:
X_gluon.drop('id', axis=1, inplace=True)

In [221]:
gluon_cat_cols = X_gluon.columns[X_gluon.dtypes == "category"].tolist()
gluon_cat_cols

['Pendapatan Keluarga',
 'Pekerjaan Orang Tua/Wali',
 'Pendidikan Orang Tua/Wali',
 'Kepemilikan Aset',
 'Lokasi Tempat Tinggal',
 'Riwayat Beasiswa atau Bantuan Pendidikan',
 'Ayah Hidup?',
 'Ibu Hidup?',
 'Status Kepemilikan Tempat Tinggal',
 'Bahan Atap Rumah',
 'Bahan Tembok Rumah',
 'Bahan Lantai Rumah',
 'Program Studi',
 'Kota',
 'Jalur Masuk',
 'Provinsi']

In [222]:
X_gluon.isnull().sum()

Jumlah Tanggungan dalam Keluarga                          0
Pengeluaran Bulanan Keluarga                              0
Status Kepesertaan Program Bantuan Pemerintah             0
Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki    0
Daya Listrik (Watt)                                       0
                                                         ..
__nlp__.semen                                             0
__nlp__.semen beton                                       0
__nlp__.semen dan                                         0
__nlp__.tembok                                            0
__nlp__._total_                                           0
Length: 68, dtype: int64

In [223]:
X_gluon[gluon_cat_cols] = X_gluon[gluon_cat_cols].astype(str)
X_gluon[gluon_cat_cols].fillna("Tidak ada", inplace=True)

In [224]:
train_raw_df.shape

(30380, 28)

In [225]:
X_gluon.shape

(30380, 68)

In [226]:
text_cols

['Bahan Atap Rumah', 'Bahan Tembok Rumah', 'Bahan Lantai Rumah']

In [227]:
X_gluon[text_cols] = X_gluon[text_cols].astype(str)
X_gluon[text_cols].fillna("Tidak ada", inplace=True)

In [228]:
X_gluon.head()

,Jumlah Tanggungan dalam Keluarga,Pengeluaran Bulanan Keluarga,Status Kepesertaan Program Bantuan Pemerintah,Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki,Daya Listrik (Watt),Luas Tanah Rumah (m2),Luas Bangunan Rumah (m2),NJOP per Meter,Rerata Pengeluaran Listrik & Air per Bulan,Total Hutang,...,__nlp__.kayu dan,__nlp__.merah,__nlp__.papan,__nlp__.pasir,__nlp__.plester,__nlp__.semen,__nlp__.semen beton,__nlp__.semen dan,__nlp__.tembok,__nlp__._total_
0,2.0,500000.0,0,2.0,900.0,144.0,126.0,103.0,450000.0,0.0,...,0,0,0,0,0,1,0,0,0,1
1,2.0,3500000.0,0,6500000.0,900.0,60.0,22.0,128.0,320000.0,45000000.0,...,0,0,0,0,1,0,0,0,0,2
2,2.0,600000.0,0,20000000.0,450.0,586.0,65.0,128000.0,-1.0,0.0,...,0,0,0,0,0,0,0,0,0,3
3,2.0,2250000.0,0,8000000.0,900.0,93.0,70.0,2176000.0,500000.0,0.0,...,0,1,0,0,0,0,0,0,0,4
4,-1.0,2310000.0,0,1000000.0,900.0,0.0,57.0,0.0,105000.0,5000000.0,...,0,0,0,0,0,0,0,0,0,3


In [229]:
X_gluon

Jumlah Tanggungan dalam Keluarga  Pengeluaran Bulanan Keluarga  \
0                                   2.0                      500000.0   
1                                   2.0                     3500000.0   
2                                   2.0                      600000.0   
3                                   2.0                     2250000.0   
4                                  -1.0                     2310000.0   
...                                 ...                           ...   
30375                               0.0                      500000.0   
30376                               0.0                      600000.0   
30377                               3.0                      450000.0   
30378                               1.0                      450000.0   
30379                               2.0                     1100000.0   

       Status Kepesertaan Program Bantuan Pemerintah  \
0                                                  0   
1                                                  0   
2                                                  0   
3                                                  0   
4                                                  0   
...                                              ...   
30375                                              0   
30376                                              1   
30377                                              0   
30378                                              0   
30379                                              0   

       Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki  \
0                                                    2.0        
1                                              6500000.0        
2                                             20000000.0        
3                                              8000000.0        
4                                              1000000.0        
...                                                  ...        
30375                                         18000000.0        
30376                                          1000000.0        
30377                                          3000000.0        
30378                                          6000000.0        
30379                                          4500000.0        

       Daya Listrik (Watt)  Luas Tanah Rumah (m2)  Luas Bangunan Rumah (m2)  \
0                    900.0                  144.0                     126.0   
1                    900.0                   60.0                      22.0   
2                    450.0                  586.0                      65.0   
3                    900.0                   93.0                      70.0   
4                    900.0                    0.0                      57.0   
...                    ...                    ...                       ...   
30375                900.0                 2700.0                      90.0   
30376                450.0                  142.0                      42.0   
30377                450.0                  450.0                      40.0   
30378                450.0                  310.0                      40.0   
30379                450.0                  441.0                      54.0   

       NJOP per Meter  Rerata Pengeluaran Listrik & Air per Bulan  \
0               103.0                                    450000.0   
1               128.0                                    320000.0   
2            128000.0                                        -1.0   
3           2176000.0                                    500000.0   
4                 0.0                                    105000.0   
...               ...                                         ...   
30375        160000.0                                     60000.0   
30376         36000.0                                     62084.0   
30377         20000.0                                         0.0   
30378         64000.0                

In [277]:
X_test = test_df[X_gluon.columns]
X_test.head()

KeyError: "['Bahan Tembok Rumah.char_count', 'Bahan Tembok Rumah.word_count', 'Bahan Tembok Rumah.capital_ratio', 'Bahan Tembok Rumah.lower_ratio', 'Bahan Tembok Rumah.digit_ratio', 'Bahan Tembok Rumah.special_ratio', 'Bahan Tembok Rumah.symbol_count. ', 'Bahan Tembok Rumah.symbol_ratio. ', 'Bahan Tembok Rumah.symbol_count./', 'Bahan Tembok Rumah.symbol_ratio./', 'Bahan Tembok Rumah.symbol_count.-', 'Bahan Tembok Rumah.symbol_ratio.-', '__nlp__.bata', '__nlp__.bata dan', '__nlp__.bata dan semen', '__nlp__.bata merah', '__nlp__.bata semen', '__nlp__.batako', '__nlp__.batu', '__nlp__.batu bata', '__nlp__.batu bata dan', '__nlp__.batu bata merah', '__nlp__.batu bata semen', '__nlp__.beton', '__nlp__.dan', '__nlp__.dan batu', '__nlp__.dan batu bata', '__nlp__.dan kayu', '__nlp__.dan semen', '__nlp__.dinding', '__nlp__.kayu', '__nlp__.kayu dan', '__nlp__.merah', '__nlp__.papan', '__nlp__.pasir', '__nlp__.plester', '__nlp__.semen', '__nlp__.semen beton', '__nlp__.semen dan', '__nlp__.tembok'

In [244]:
X_test = X_test.copy()
X_test[gluon_cat_cols] = X_test[gluon_cat_cols].astype(str)
X_test[gluon_cat_cols].fillna("Tidak ada", inplace=True)
# X_test[cat_cols].fillna("Yang Lain", inplace=True)
X_test.fillna(-1, inplace=True)

In [245]:
X_test.isnull().sum()

Jumlah Tanggungan dalam Keluarga                          0
Pengeluaran Bulanan Keluarga                              0
Status Kepesertaan Program Bantuan Pemerintah             0
Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki    0
Daya Listrik (Watt)                                       0
                                                         ..
__nlp__.semen                                             0
__nlp__.semen beton                                       0
__nlp__.semen dan                                         0
__nlp__.tembok                                            0
__nlp__._total_                                           0
Length: 68, dtype: int64

In [10]:
train_raw_df = pd.read_csv("dataset/train.csv")
train_df = pd.read_csv("train_gluon_no_clean.csv")
test_df = pd.read_csv("test_gluon_no_clean.csv")

In [11]:
test_df.head()

,id,Jumlah Tanggungan dalam Keluarga,Pengeluaran Bulanan Keluarga,Status Kepesertaan Program Bantuan Pemerintah,Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki,Daya Listrik (Watt),Luas Tanah Rumah (m2),Luas Bangunan Rumah (m2),NJOP per Meter,Rerata Pengeluaran Listrik & Air per Bulan,...,__nlp__.sudah,__nlp__.tanah,__nlp__.tembok,__nlp__.tembok bata,__nlp__.tembok batu,__nlp__.tembok dan,__nlp__.tidak,__nlp__.triplek,__nlp__.yang,__nlp__._total_
0,80884,0.0,2500000.0,0,6000000.0,450.0,472.0,54.0,48000.0,105000.0,...,0,0,0,0,0,0,0,0,0,3
1,44867,1.0,4000000.0,1,8000000.0,900.0,-1.0,60.0,91510000.0,175000.0,...,0,0,0,0,0,0,0,0,0,3
2,110878,1.0,800000.0,0,25000000.0,1105.0,216.0,64.0,48000.0,120000.0,...,0,1,1,0,0,0,0,0,0,7
3,146013,2.0,800000.0,0,3.0,1300.0,1460.0,80.0,48000.0,260000.0,...,0,0,0,0,0,0,0,0,0,1
4,145148,1.0,2500000.0,0,1.0,900.0,72.0,72.0,0.0,-1.0,...,0,0,0,0,0,0,0,0,0,1


In [12]:
X_gluon, y = train_df, train_raw_df["Golongan UKT"]

In [73]:
len(y)

30380

In [74]:
X_gluon.isnull().sum()

id                                                        0
Jumlah Tanggungan dalam Keluarga                          0
Pengeluaran Bulanan Keluarga                              0
Status Kepesertaan Program Bantuan Pemerintah             0
Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki    0
                                                         ..
__nlp__.tembok dan                                        0
__nlp__.tidak                                             0
__nlp__.triplek                                           0
__nlp__.yang                                              0
__nlp__._total_                                           0
Length: 134, dtype: int64

In [66]:
X_gluon.fillna(-1, inplace=True)
X_gluon.isnull().sum().sum()

0

In [67]:
test_df.fillna(-1, inplace=True)
test_df.isnull().sum().sum()

0

In [85]:
class_weights_dict = dict()
n, num_class = train_raw_df.shape[0], len(train_raw_df['Golongan UKT'].unique())
for label in train_raw_df['Golongan UKT'].unique():
    n_l = train_raw_df[train_raw_df['Golongan UKT'] == label].shape[0]
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

sample_weights = train_raw_df['Golongan UKT'].map(class_weights_dict).values
sample_weights

array([0.68337232, 0.74636399, 0.57390056, ..., 0.68337232, 0.68337232,
       0.68337232])

In [103]:
n_folds = 5
dir_name = "output"

# Train CatBoost model
cb_score, cb_oof, cb_test = multiclass_oof_preds(
    CatBoostClassifier,
    "CatBoost",
    X_gluon,
    y,
    test_df,
    metric_score=f1_score,
    metric_score_args={"average": "macro"},
    model_args={
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        # "max_bin": 254,
        "task_type": "GPU",
        "devices": "0",
        "verbose": 200,
        "random_state": 42,
        # "cat_features": cat_cols,
        # "text_features": text_cols,
        # "class_weights": class_weights_dict,
    },
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

Learning rate set to 0.124441
0:	learn: 0.4587572	total: 7.58ms	remaining: 7.57s
200:	learn: 0.6385551	total: 1.31s	remaining: 5.2s
400:	learn: 0.6961129	total: 2.63s	remaining: 3.94s
600:	learn: 0.7475090	total: 3.96s	remaining: 2.63s
800:	learn: 0.7897973	total: 5.38s	remaining: 1.34s
999:	learn: 0.8257104	total: 7s	remaining: 0us
Learning rate set to 0.124441
0:	learn: 0.4614270	total: 8.96ms	remaining: 8.96s
200:	learn: 0.6398590	total: 1.42s	remaining: 5.66s
400:	learn: 0.7057235	total: 2.7s	remaining: 4.04s
600:	learn: 0.7555934	total: 4.04s	remaining: 2.68s
800:	learn: 0.7973006	total: 5.32s	remaining: 1.32s
999:	learn: 0.8344925	total: 6.63s	remaining: 0us
Learning rate set to 0.124441
0:	learn: 0.4613418	total: 8.25ms	remaining: 8.24s
200:	learn: 0.6407924	total: 1.34s	remaining: 5.34s
400:	learn: 0.6967883	total: 2.64s	remaining: 3.94s
600:	learn: 0.7445832	total: 3.93s	remaining: 2.61s
800:	learn: 0.7856189	total: 5.27s	remaining: 1.31s
999:	learn: 0.8269046	total: 6.64s	rem

In [10]:
n_folds = 5
dir_name = "output"

# Train CatBoost model
cb_score, cb_oof, cb_test = multiclass_oof_preds(
    CatBoostClassifier,
    "CatBoost",
    X_gluon,
    y,
    test_df,
    metric_score=f1_score,
    metric_score_args={"average": "macro"},
    model_args={
        "max_bin": 254,
        "task_type": "GPU",
        "devices": "0",
        "verbose": 200,
        "random_state": 42,
        # "cat_features": cat_cols,
        # "text_features": text_cols,
    },
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

Learning rate set to 0.124441
0:	learn: 1.8289209	total: 47.1ms	remaining: 47s
200:	learn: 0.9219874	total: 1.49s	remaining: 5.93s
400:	learn: 0.8321855	total: 2.89s	remaining: 4.32s
600:	learn: 0.7513312	total: 4.32s	remaining: 2.87s
800:	learn: 0.6834216	total: 5.76s	remaining: 1.43s
999:	learn: 0.6216260	total: 7.19s	remaining: 0us
Learning rate set to 0.124441
0:	learn: 1.8288536	total: 8.87ms	remaining: 8.86s
200:	learn: 0.9273860	total: 1.44s	remaining: 5.73s
400:	learn: 0.8331038	total: 2.85s	remaining: 4.25s
600:	learn: 0.7546465	total: 4.27s	remaining: 2.84s
800:	learn: 0.6860284	total: 5.7s	remaining: 1.42s
999:	learn: 0.6223216	total: 7.14s	remaining: 0us
Learning rate set to 0.124441
0:	learn: 1.8285792	total: 7.66ms	remaining: 7.65s
200:	learn: 0.9237536	total: 1.43s	remaining: 5.68s
400:	learn: 0.8350656	total: 2.85s	remaining: 4.26s
600:	learn: 0.7609621	total: 4.3s	remaining: 2.85s
800:	learn: 0.6942333	total: 5.7s	remaining: 1.42s
999:	learn: 0.6368921	total: 7.1s	rema

In [11]:
X_gluon

id  Jumlah Tanggungan dalam Keluarga  Pengeluaran Bulanan Keluarga  \
0      127748                               2.0                      500000.0   
1       90205                               2.0                     3500000.0   
2       82795                               2.0                      600000.0   
3      108947                               2.0                     2250000.0   
4       89985                              -1.0                     2310000.0   
...       ...                               ...                           ...   
30375  147514                               0.0                      500000.0   
30376   62832                               0.0                      600000.0   
30377   88943                               3.0                      450000.0   
30378  149599                               1.0                      450000.0   
30379   82699                               2.0                     1100000.0   

       Status Kepesertaan Program Bantuan Pemerintah  \
0                                                  0   
1                                                  0   
2                                                  0   
3                                                  0   
4                                                  0   
...                                              ...   
30375                                              0   
30376                                              1   
30377                                              0   
30378                                              0   
30379                                              0   

       Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki  \
0                                                    2.0        
1                                              6500000.0        
2                                             20000000.0        
3                                              8000000.0        
4                                              1000000.0        
...                                                  ...        
30375                                         18000000.0        
30376                                          1000000.0        
30377                                          3000000.0        
30378                                          6000000.0        
30379                                          4500000.0        

       Daya Listrik (Watt)  Luas Tanah Rumah (m2)  Luas Bangunan Rumah (m2)  \
0                    900.0                  144.0                     126.0   
1                    900.0                   60.0                      22.0   
2                    450.0                  586.0                      65.0   
3                    900.0                   93.0                      70.0   
4                    900.0                    0.0                      57.0   
...                    ...                    ...                       ...   
30375                900.0                 2700.0                      90.0   
30376                450.0                  142.0                      42.0   
30377                450.0                  450.0                      40.0   
30378                450.0                  310.0                      40.0   
30379                450.0                  441.0                      54.0   

       NJOP per Meter  Rerata Pengeluaran Listrik & Air per Bulan  ...  \
0               103.0                                    450000.0  ...   
1               128.0                                    320000.0  ...   
2            128000.0                                        -1.0  ...   
3           2176000.0                                    500000.0  ...   
4                 0.0                                    105000.0  ...   
...               ...                                         ...  ...   
30375        160000.0                                     60000.0  ...   
30376         36000.0                      

In [78]:
n_folds = 5
dir_name = "output"

# Train LightGBM model
lgbm_score, lgbm_oof, lgbm_test = multiclass_oof_preds(
    LGBMClassifier,
    "LightGBM",
    X_gluon,
    y,
    test_df,
    metric_score=f1_score,
    metric_score_args={"average": "macro"},
    model_args={
        "device": "gpu",
        # "n_estimators": 1000,
        # "verbose": -1,
        "random_state": 42,
        "device_type": "gpu",
        "class_weight": class_weights_dict,
    },
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

[LightGBM] [Warning] device_type is set=gpu, device=gpu will be ignored. Current value: device_type=gpu
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] device_type is set=gpu, device=gpu will be ignored. Current value: device_type=gpu
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3904
[LightGBM] [Info] Number of data points in the train set: 24304, number of used features: 134
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 3060 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 32 dense feature groups (0.74 MB) transferred to GPU in 0.004070 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -2.079602
[LightGBM] [Info] Start training from score -2.079467
[LightGBM] [Info] Start training from score -2.079359
[LightGBM] 

In [256]:
# X_xgb = X_gluon.copy()
# for col in cat_cols:
#     X_xgb.loc[X_xgb[col] == 'nan', col] = -1
#     X_xgb[col] = X_xgb[col].astype(int)

In [263]:
# X_test_xgb = X_gluon.copy()
# for col in cat_cols:
#     X_test_xgb.loc[X_test_xgb[col] == 'nan', col] = -1
#     X_test_xgb[col] = X_test_xgb[col].astype(int)

In [262]:
X_xgb.info()

<class 'autogluon.core.dataset.TabularDataset'>
RangeIndex: 30380 entries, 0 to 30379
Data columns (total 68 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   Jumlah Tanggungan dalam Keluarga                        30380 non-null  float64
 1   Pengeluaran Bulanan Keluarga                            30380 non-null  float64
 2   Status Kepesertaan Program Bantuan Pemerintah           30380 non-null  int32  
 3   Jumlah (Perkiraan Harga Pasar) Kendaraan yang Dimiliki  30380 non-null  float64
 4   Daya Listrik (Watt)                                     30380 non-null  float64
 5   Luas Tanah Rumah (m2)                                   30380 non-null  float64
 6   Luas Bangunan Rumah (m2)                                30380 non-null  float64
 7   NJOP per Meter                                          30380 non-null  float64
 8   Rerata Pengeluaran Listrik

In [88]:
n_folds = 5
dir_name = "output"

# Train XGBoost model
xgb_score, xgb_oof, xgb_test = multiclass_oof_preds(
    XGBClassifier,
    "XGBoost",
    X_gluon,
    y,
    test_df,
    metric_score=f1_score,
    metric_score_args={"average": "macro"},
    model_args={
        "tree_method": "gpu_hist",
        "verbosity": "0",
        # "text_features": text_cols,
    },
    # fitting_args={
    #     "sample_weight": sample_weights,
    # },
    n_folds=n_folds,
    dir_name=dir_name,
    verbose=True,
)

Model: XGBoost
Average Metric: 45.79
Training time (mins): 0.05



In [118]:
# Join oof preds
used_models = ["CatBoost", "XGBoost", "LightGBM"]

oof_df = pd.DataFrame(index=np.arange(len(y)))
for i in used_models:
    with open(f"output/{i}_oof_preds.pkl", "rb") as f:
        oof_preds = pkl.load(f)
    df = pd.DataFrame({i: oof_preds.tolist()})
    oof_df = pd.concat([oof_df, df], axis=1)  # axis = 1, add on another column

# Join test preds
test_preds_df = pd.DataFrame(index=np.arange(len(test_df)))
for i in used_models:
    with open(f"output/{i}_test_preds.pkl", "rb") as f:
        test_preds = pkl.load(f)
    df = pd.DataFrame({i: test_preds.tolist()})
    test_preds_df = pd.concat([test_preds_df, df], axis=1)

display(oof_df.head(3))
display(test_preds_df.head(3))

,CatBoost,XGBoost,LightGBM
0,"[0.9900805663803119, 0.0002808090690795848, 0....","[0.9997448325157166, 1.2856578450737288e-06, 6...","[0.9999475254040506, 1.5146482652932436e-06, 2..."
1,"[0.0011100662988913074, 6.908219144892048e-05,...","[3.088411904172972e-05, 0.00028301560087129474...","[2.8899458277685734e-05, 0.0001604475635228986..."
2,"[0.004045152658126354, 0.00015420485054690086,...","[0.00032437845948152244, 0.0001360151218250394...","[0.0001488451902917201, 0.00036368814383749235..."


,CatBoost,XGBoost,LightGBM
0,"[0.0034682886253137608, 0.00046366667212708663...","[2.8558628855535062e-05, 0.0003910922223440138...","[2.4997826618104513e-05, 0.0006994655892793814..."
1,"[0.00032398512225562066, 0.0008470369518981236...","[4.903653325527557e-05, 0.002186706682550721, ...","[2.0687883962734072e-05, 0.0027072282498008223..."
2,"[0.0032654642642752753, 4.850226748028145e-05,...","[0.00028037144693371374, 4.1750420905373176e-0...","[4.657213240123928e-05, 0.00021522152907417296..."


In [119]:
def multiclass_score(y_true, y_pred, **kwargs):
    return f1_score(
        y_true, np.argmax(np.array(y_pred.tolist()), axis=1), **kwargs
    )

In [120]:
# Evaluate oof preds
scores = {}
for col in oof_df.columns:
    scores[col] = multiclass_score(y, oof_df[col], average="macro")

# Sort scores
scores = {
    k: v for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True)
}

# Sort oof_df and test_preds
oof_df = oof_df[list(scores.keys())]
test_preds_df = test_preds_df[list(scores.keys())]

scores

{'CatBoost': 0.46746169657756875,
 'LightGBM': 0.4637679238138436,
 'XGBoost': 0.45792106643509256}

In [107]:
# Initialise
STOP = False
current_best_ensemble = np.array(oof_df.iloc[:, 0].tolist())
current_best_test_preds = np.array(test_preds_df.iloc[:, 0].tolist())
MODELS = oof_df.iloc[:, 1:]  # the second, third, ... best models
weight_range = np.arange(
    0.01, 0.51, 0.01
)  # or with negative weights: np.arange(-0.5,0.51,0.01)
history = [multiclass_score(y, current_best_ensemble, average="macro")]
i = 0

# Hill climbing
while not STOP:
    i += 1
    potential_new_best_cv_score = multiclass_score(
        y, current_best_ensemble, average="macro"
    )
    k_best, wgt_best = None, None
    for k in MODELS:
        for wgt in weight_range:
            potential_ensemble = (1 - wgt) * np.array(
                current_best_ensemble.tolist()
            ) + wgt * np.array(MODELS[k].tolist())
            cv_score = multiclass_score(y, potential_ensemble, average="macro")
            if cv_score > potential_new_best_cv_score:
                potential_new_best_cv_score = cv_score
                k_best, wgt_best = k, wgt

    if k_best is not None:
        current_best_ensemble = (
            1 - wgt_best
        ) * current_best_ensemble + wgt_best * np.array(MODELS[k_best].tolist())
        current_best_test_preds = (
            1 - wgt_best
        ) * current_best_test_preds + wgt_best * np.array(
            test_preds_df[k_best].tolist()
        )
        MODELS.drop(
            k_best, axis=1, inplace=True
        )  # model kedua yang sudah ditambahkan, didrop, tidak dipertimbangkan lagi
        if MODELS.shape[1] == 0:
            STOP = True
        print(
            f"Iteration: {i}, Model added: {k_best}, Best weight: {wgt_best:.2f}, Best metric: {potential_new_best_cv_score:.5f}"
        )
        history.append(potential_new_best_cv_score)
    else:
        STOP = True

Iteration: 1, Model added: LightGBM, Best weight: 0.35, Best metric: 0.47535
Iteration: 2, Model added: XGBoost, Best weight: 0.01, Best metric: 0.47539


In [108]:
current_best_test_preds

array([[2.24079106e-03, 5.44645252e-04, 9.35462695e-02, ...,
        2.60037557e-01, 1.18674929e-01, 1.44167980e-02],
       [2.16143143e-04, 1.50498993e-03, 3.99736820e-03, ...,
        3.37753488e-01, 2.48978690e-01, 1.00870167e-01],
       [2.12026721e-03, 1.06202973e-04, 1.12546159e-02, ...,
        3.28167431e-01, 1.38487726e-01, 3.41625507e-02],
       ...,
       [5.32981166e-03, 1.06837410e-04, 2.24753033e-03, ...,
        3.93862936e-01, 3.09563973e-01, 1.40547714e-01],
       [8.04588537e-04, 2.15730106e-03, 2.85531897e-02, ...,
        2.12356646e-01, 1.08008580e-01, 5.92350732e-02],
       [5.45966247e-03, 3.28872128e-04, 1.70934312e-02, ...,
        1.67486845e-01, 4.50999985e-02, 6.26658055e-03]])

In [109]:
len(current_best_test_preds)

7595

In [57]:
len(pd.read_csv("dataset/test.csv")["id"])

7595

In [110]:
submission_df = pd.DataFrame(
    {
        "id": pd.read_csv("dataset/test.csv")["id"],
        "Golongan UKT": np.argmax(current_best_test_preds, axis=1),
    }
)
submission_df.head()

,id,Golongan UKT
0,80884,4
1,44867,5
2,110878,4
3,146013,4
4,145148,0


In [111]:
submission_df.to_csv("last3.csv", index=False)